# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds (Kaggle)

Bản chạy **một mạch** (Save & Run All) trên Kaggle: GPU + Internet bật. Mã nguồn các module (`dataset, model, losses, train,
inference, benchmark, final, checks, tables, report_figs, kaggle_run`) nằm trong ô đầu (nén base64, giải nén ra
`/kaggle/working/submissions/<mssv>_<slug>/code/`), `eval.py` gốc ở `/kaggle/working/eval.py` (không sửa).

**Quy tắc:** mọi lựa chọn chốt trên **val** bằng quy tắc ghi trong `kaggle_run.py` (không chọn tay, không nhìn test);
**test chỉ chạy một lượt mỗi seed ở Bước 4**. `SMOKE = True` chỉ để chạy thử môi trường, số liệu đó không dùng để nộp.

In [ ]:
# ===== CẤU HÌNH =====
import os
MSSV, SLUG = "2A202602985", "ngo_duc_chung"
SMOKE = os.environ.get("LAB_SMOKE", "0") == "1"   # 1 = chạy thử 1 epoch / vài batch để kiểm tra môi trường; KHÔNG dùng số liệu này để nộp
IMG = 224            # độ phân giải huấn luyện và đánh giá
EPOCHS = 10          # số epoch của công thức nền T00 (GUIDE: 10-15); ablation G thử 20
BUDGET_ABL_H = 4.5   # giờ kể từ lúc bắt đầu: quá mốc này thì bỏ các ablation còn lại để dành giờ cho chung kết
BUDGET_L2_H = 8.0    # giờ: quá mốc này thì bỏ các lần chạy còn lại của ma trận Lab #2 (chạy SAU chung kết)
SEEDS = (0, 1, 2)    # seed của chung kết và mốc

In [ ]:
# ENV: giải nén mã nguồn, cài thư viện, kiểm tra GPU
import os, sys, io, base64, tarfile, subprocess, time, shutil, platform
T_START = time.time()
ON_KAGGLE = os.path.isdir("/kaggle/working")
ROOT = ("/kaggle/working" if ON_KAGGLE else "/content/work") + ("_smoke" if SMOKE else "")
os.makedirs(ROOT, exist_ok=True)
SUB = f"{ROOT}/submissions/{MSSV}_{SLUG}"
DRIVE_OUT = None if (SMOKE or ON_KAGGLE) else "/content/drive/MyDrive/K4_Day2_DeepWeeds_out"   # chỉ có tác dụng khi Drive đã được mount (kiểm tra mỗi lần sao lưu)
os.makedirs(f"{SUB}/code", exist_ok=True)
BUNDLE = "..."
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz") as tf:
    tf.extractall(ROOT)
subprocess.run("pip -q install timm openpyxl", shell=True, check=True)
import torch, timm
sys.path.insert(0, ROOT); sys.path.insert(0, f"{SUB}/code"); os.chdir(SUB)
print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert torch.cuda.is_available(), "Bật Accelerator (GPU) và Internet trong Session options rồi chạy lại"

In [ ]:
# ENV: tải DeepWeeds (Zenodo 7939060) vào ổ tạm, kiểm tra MD5, tải nhãn fold 0 nguyên bản từ GitHub của tác giả
import hashlib, glob, zipfile
DATA = "/kaggle/temp/data" if os.path.isdir("/kaggle/temp") else "/content/data"; os.makedirs(f"{DATA}/labels", exist_ok=True)
MD5 = "b7b30f96d466fba86016aa5a26606e0f"
ZIP = f"{DATA}/images.zip"
URL = "https://zenodo.org/records/7939060/files/images.zip?download=1"
def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
    return h.hexdigest()
for attempt in range(5):
    if os.path.exists(ZIP) and md5(ZIP) == MD5: break
    print("tải images.zip, lần", attempt + 1, flush=True)
    subprocess.run(["curl", "-L", "-sS", "--retry", "5", "--retry-delay", "5", "-C", "-", "-o", ZIP, URL])
    if os.path.exists(ZIP) and md5(ZIP) != MD5: os.remove(ZIP)   # hỏng thì tải lại từ đầu
assert os.path.exists(ZIP) and md5(ZIP) == MD5, "MD5 images.zip sai sau 5 lần tải"
print("MD5 OK")
with zipfile.ZipFile(ZIP) as z: z.extractall(DATA)
BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for n in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    subprocess.run(["curl", "-L", "-sS", "--retry", "3", "-o", f"{DATA}/labels/{n}.csv", f"{BASE}/{n}.csv"], check=True)
jpgs = glob.glob(f"{DATA}/**/*.jpg", recursive=True)
IMAGES_DIR = os.path.dirname(jpgs[0]); LABELS_DIR = f"{DATA}/labels"
print(len(jpgs), "ảnh (kỳ vọng 17509) |", IMAGES_DIR)
assert len(jpgs) == 17509

In [ ]:
# Đường dẫn, cfg(), run_many() dùng cho mọi bước
import json, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import dataset as D, model as M, losses as L, train as T, inference as I, benchmark as B
import checks, final as FN, tables as TB, report_figs as RF, kaggle_run as K
ev = T.ev
K.STRICT = SMOKE                                           # chạy thử thì lỗi dừng ngay; chạy thật thì ghi log rồi đi tiếp
K.LOG_FILE = pathlib.Path(f"{SUB}/logs/progress.log")
OUT, PRED, CURVE, TAB = (f"{SUB}/{d}" for d in ("run_logs", "predictions", "curves", "tables"))
for d in (OUT, PRED, CURVE, TAB): os.makedirs(d, exist_ok=True)
DEV = M.get_device()
EXTRA = dict(epochs=1, debug_batches=3) if SMOKE else dict(epochs=EPOCHS)
NW = min(4, os.cpu_count() or 2)

def cfg(**kw):
    base = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=OUT, pred_dir=PRED, curves_dir=CURVE,
                num_workers=NW, cache_images=True, img_size=IMG)
    base.update(EXTRA); base.update(kw)
    return T.Config(**base)

def hours(): return (time.time() - T_START) / 3600

def run_many(exps, budget_h=None):
    """Mỗi mục (exp_id, override). Chạy lại được sau khi ngắt (skip_if_done + resume). Hết ngân sách thì bỏ qua."""
    out = []
    for e, kw in exps:
        if budget_h is not None and hours() > budget_h:
            K.log(f"BỎ QUA {e}: đã quá ngân sách {budget_h} giờ"); continue
        ok, r = K.stage(f"train {e} {kw}", T.run, cfg(exp_id=e, **kw))
        if ok: out.append(r)
        K.backup(SUB, DRIVE_OUT)
    return out

K.log(f"bắt đầu | nền tảng={'Kaggle' if ON_KAGGLE else 'Colab'} | Drive backup={DRIVE_OUT} | workers={NW} | SMOKE={SMOKE} | IMG={IMG} | GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else DEV}")

In [ ]:
# Test CPU của code (không cần GPU/dữ liệu thật)
with K.guard("unit test"):
    if os.environ.get("LAB_SKIP_TESTS") == "1": raise RuntimeError("bỏ qua unit test ở lần khởi động lại (đã chạy ở lần khởi động đầu, xem log)")
    r = subprocess.run([sys.executable, "-m", "unittest", "test_code"], cwd=f"{SUB}/code", capture_output=True, text=True)
    print(r.stderr[-400:])

## Bước 0 — EDA và kiểm tra pipeline

In [ ]:
with K.guard("Bước 0: kiểm tra split"):
    train_df, val_df, test_df = D.load_split(LABELS_DIR, fold=0)
    split_info = D.check_split(train_df, val_df, test_df, IMAGES_DIR)   # dừng nếu vi phạm S1-S4
    json.dump(split_info, open(f"{TAB}/split_check.json", "w"), indent=1)

In [ ]:
with K.guard("Bước 0: EDA"):
    allv = pd.read_csv(f"{LABELS_DIR}/labels.csv")
    counts = pd.DataFrame({"all": allv["Label"].value_counts().sort_index(),
                           "train": train_df["Label"].value_counts().sort_index(),
                           "val": val_df["Label"].value_counts().sort_index(),
                           "test": test_df["Label"].value_counts().sort_index()})
    counts.index = D.CLASS_NAMES
    print(counts, "\ntổng", counts["all"].sum(), "| tỉ lệ lớp nhiều nhất / ít nhất:",
          round(counts["all"].max() / counts["all"].min(), 2))
    counts.to_csv(f"{TAB}/class_counts.csv")
    ax = counts[["train", "val", "test"]].plot.bar(stacked=True, figsize=(9, 4), title="Số ảnh mỗi lớp theo tập")
    ax.figure.tight_layout(); ax.figure.savefig(f"{TAB}/eda_class_distribution.png", dpi=130); plt.show()
    from PIL import Image
    fig, axs = plt.subplots(9, 4, figsize=(8, 18))
    for c in range(9):
        rows = train_df[train_df.Label == c].sample(4, random_state=0)
        for a, f in zip(axs[c], rows.Filename):
            a.imshow(Image.open(f"{IMAGES_DIR}/{f}")); a.axis("off")
        axs[c][0].set_title(D.CLASS_NAMES[c], fontsize=8, loc="left")
    fig.tight_layout(); fig.savefig(f"{TAB}/eda_samples.png", dpi=110); plt.show()

In [ ]:
with K.guard("Bước 0: kiểm tra pipeline"):
    # loss đầu ≈ ln 9 = 2.197; overfit một batch nhỏ; ảnh sau augmentation khớp nhãn
    c0 = cfg(backbone="resnet50")
    PIPE = {"initial_loss": checks.initial_loss(c0)}
    trace = checks.overfit_one_batch(c0, n=16, steps=60)
    PIPE["overfit_first_last"] = [trace[0], trace[-1]]
    json.dump(PIPE, open(f"{TAB}/pipeline_checks.json", "w"), indent=1, default=str)
    for aug, mix in [("basic", None), ("geo", None), ("color", None), ("trivial", None), ("basic", "cutmix"), ("basic", "mixup")]:
        fig = checks.show_augmented(cfg(aug=aug), n=8, mix=mix)
        fig.savefig(f"{TAB}/augmented_{aug}{'_' + mix if mix else ''}.png", dpi=100); plt.show()

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00`, cùng seed 0, mỗi backbone một lần chạy (`B01`...`B07`).

In [ ]:
BACKBONES = [b for b in K.BACKBONES if b[0] in ("B01", "B04", "B07")] if SMOKE else K.BACKBONES
with K.guard("Bước 1: huấn luyện backbone"):
    run_many([(e, dict(backbone=n, seed=0)) for e, n in BACKBONES])
    runs = TB.collect_summaries(OUT)
    bb = runs[runs.exp_id.isin([e for e, _ in BACKBONES])].copy()
    bb.to_csv(f"{TAB}/backbones_raw.csv", index=False)
    print(bb[["exp_id", "backbone", "pretrained_tag", "params_m", "gmacs", "val_macro_f1", "val_top1", "sec_per_epoch"]])

In [ ]:
with K.guard("Bước 1: độ trễ backbone"):
    lat = []
    for e, n in BACKBONES:
        m = T.load_best(cfg(exp_id=e, backbone=n, seed=0), DEV)
        for batch in (1, 32):
            for dt in ("fp32", "amp"):
                r = B.latency_report(m, batch, IMG, dt, DEV.type, warmup=20, iters=100)
                lat.append({"exp_id": e, "backbone": n, "bn_fused": False, **r})
        del m
        if DEV.type == "cuda": torch.cuda.empty_cache()
    lat = pd.DataFrame(lat); lat.to_csv(f"{TAB}/latency_backbones.csv", index=False)
    b1 = lat[(lat.batch == 1) & (lat.dtype == "fp32")].set_index("exp_id")["p50"].rename("lat_b1_fp32_ms")
    backbones_tbl = bb.merge(b1, left_on="exp_id", right_index=True, how="left")
    backbones_tbl["ghi chú"] = "1 seed; công thức nền T00"
    BEST_BB = K.pick_backbone(backbones_tbl)
    print("backbone được chọn (macro-F1 val cao nhất):", BEST_BB)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(backbones_tbl.lat_b1_fp32_ms, backbones_tbl.val_macro_f1, s=backbones_tbl.params_m * 6)
    for _, r in backbones_tbl.iterrows():
        ax.annotate(r.backbone, (r.lat_b1_fp32_ms, r.val_macro_f1), fontsize=7)
    ax.set(xlabel="độ trễ batch-1 FP32 p50 (ms)", ylabel="macro-F1 val", title="Backbone: chất lượng vs tốc độ (kích thước = #params)")
    fig.tight_layout(); fig.savefig(f"{TAB}/backbones_tradeoff.png", dpi=130); plt.show()
    backbones_tbl[["exp_id", "backbone", "pretrained_tag", "params_m", "gmacs", "val_macro_f1", "val_top1", "sec_per_epoch", "lat_b1_fp32_ms"]]

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` một yếu tố. Đo nhiễu bằng `T00` × 3 seed; yếu tố nào có Δ lớn hơn std của T00 mới được xem là thắng.
Tổ hợp `T18` ghép yếu tố thắng của từng nhóm (tham lam theo trục). Công thức chung kết = tốt nhất theo val trong {T00, T18, yếu tố thắng đơn}.

In [ ]:
BB = BEST_BB
base = dict(backbone=BB)
AX = [a for a in K.AXES if (a[0] in ("T03", "T09", "T16") if SMOKE else True)]
with K.guard("Bước 2: T00 x3 và ablation"):
    run_many([("T00", dict(base, seed=s)) for s in SEEDS])
    run_many([(e, dict(base, seed=0, **kw)) for e, _, _, kw in AX], budget_h=BUDGET_ABL_H)

In [ ]:
with K.guard("Bước 2: bảng ablation"):
    runs = TB.collect_summaries(OUT)
    t00 = runs[runs.exp_id == "T00"]
    noise = K.noise_threshold(t00.val_macro_f1.to_numpy())
    t00_ref = float(t00[t00.seed == 0].val_macro_f1.iloc[0])
    names = ev.load_names(f"{LABELS_DIR}/labels.csv"); hard = ev.hard_class_indices(names)
    def hard_f1(exp_id, seed=0):
        p = ev.read_pred(f"{PRED}/{exp_id}_seed{seed}_val.csv")
        m = ev.compute_metrics(p.y_true, p.y_pred, p.probs)
        return {f"F1 val {n}": float(m["f1"][i]) for n, i in hard.items()}
    print(f"T00: macro-F1 val {t00.val_macro_f1.mean():.4f} ± {t00.val_macro_f1.std(ddof=1):.4f} (n={len(t00)} seed); ngưỡng nhiễu dùng = {noise:.4f}")
    def verdict(d):
        return "vượt nhiễu (>2σ)" if abs(d) > 2 * noise else ("gần ngưỡng (1-2σ)" if abs(d) > noise else "không phân biệt được với nhiễu")
    rows = [{"exp_id": "T00", "backbone": BB, "trục": "nền", "khác T00": "(công thức nền)", "seed": 0,
             "val_macro_f1": t00_ref, "val_top1": float(t00[t00.seed == 0].val_top1.iloc[0]), "delta_vs_T00": 0.0,
             "delta/σ": 0.0, "kết luận": "mốc", **hard_f1("T00")}]
    for e, ax_, desc, kw in AX:
        r = runs[(runs.exp_id == e) & (runs.seed == 0)]
        if r.empty: continue
        r = r.iloc[0]; d = float(r.val_macro_f1 - t00_ref)
        rows.append({"exp_id": e, "backbone": BB, "trục": ax_, "khác T00": desc, "seed": 0,
                     "val_macro_f1": r.val_macro_f1, "val_top1": r.val_top1, "delta_vs_T00": d,
                     "delta/σ": d / noise, "kết luận": verdict(d), **hard_f1(e)})
    training_tbl = pd.DataFrame(rows)
    COMBO, CHOSEN, BEST_SINGLE = K.pick_winners(training_tbl, noise)
    K.log(f"yếu tố thắng (Δ > {noise:.4f}): {CHOSEN} -> COMBO={COMBO} | thắng đơn tốt nhất: {BEST_SINGLE}")
    print(training_tbl.drop(columns=["backbone"]))

In [ ]:
with K.guard("Bước 2: T18 (kết hợp) và chốt công thức"):
    if COMBO:
        run_many([("T18", dict(base, seed=0, **COMBO))])
    runs = TB.collect_summaries(OUT)
    FINAL_SRC, FINAL_RECIPE = K.pick_final_recipe(runs, COMBO, BEST_SINGLE, base_f1=t00_ref)
    K.log(f"công thức chung kết (chọn theo val): nguồn={FINAL_SRC} recipe={FINAL_RECIPE}")
    t18 = runs[(runs.exp_id == "T18") & (runs.seed == 0)]
    if len(t18):
        d = float(t18.iloc[0].val_macro_f1 - t00_ref)
        training_tbl = pd.concat([training_tbl, pd.DataFrame([{
            "exp_id": "T18", "backbone": BB, "trục": "kết hợp", "khác T00": "ghép: " + ", ".join(CHOSEN) + " " + str(COMBO),
            "seed": 0, "val_macro_f1": t18.iloc[0].val_macro_f1, "val_top1": t18.iloc[0].val_top1, "delta_vs_T00": d,
            "delta/σ": d / noise, "kết luận": verdict(d), **hard_f1("T18")}])], ignore_index=True)
    training_tbl["ghi chú"] = "1 seed; " + training_tbl["kết luận"]
    training_tbl.to_csv(f"{TAB}/training_ablation.csv", index=False)
    RF.ablation_figure(training_tbl[training_tbl.exp_id != "T00"], noise, f"{TAB}/ablation_T.png"); plt.show()
    training_tbl

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**, không huấn luyện lại. Model: checkpoint tốt nhất của công thức đã chốt ở Bước 2.

In [ ]:
with K.guard("Bước 3: I00-I03"):
    BEST = cfg(exp_id=FINAL_SRC, backbone=BB, seed=0)
    model = T.load_best(BEST, DEV)
    vdf = val_df.head(256) if SMOKE else val_df
    def vloader(size): return D.make_loader(vdf, IMAGES_DIR, D.build_transforms(False, size), 128, False, num_workers=2)
    S256 = int(round(IMG * 256 / 224))
    def score(probs, y):
        m = ev.compute_metrics(y, probs.argmax(1), probs)
        return {"macro_f1": m["macro_f1"], "top1": m["top1"], "ece": m["ece"], "nll": m["nll"]}
    inf_rows = []
    def add(eid, name, k, probs, y, note=""):
        inf_rows.append({"exp_id": eid, "phương pháp": name, "mô hình": f"{FINAL_SRC} ({BB})", "K": k, **score(probs, y), "ghi chú": note})
    _, yv, z0 = I.predict_logits(model, vloader(IMG), DEV)
    add("I00", "1 view (mốc)", 1, I.apply_temperature(z0, 1.0), yv)
    _, _, zs = I.predict_multiview(model, vloader(IMG), DEV, [I.view_identity, I.view_hflip])
    add("I01", "TTA lật ngang", 2, I.aggregate_views(zs, "prob"), yv)
    _, _, zc = I.predict_multiview(model, vloader(S256), DEV, lambda x: I.views_multicrop(x, IMG))
    add("I02a", "TTA 5 crop", 5, I.aggregate_views(zc, "prob"), yv)
    _, _, zcf = I.predict_multiview(model, vloader(S256), DEV, lambda x: I.views_multicrop(x, IMG, flip=True))
    add("I02b", "TTA 5 crop + lật", 10, I.aggregate_views(zcf, "prob"), yv)
    add("I03a", "TTA 5 crop, gộp prob", 5, I.aggregate_views(zc, "prob"), yv)
    add("I03b", "TTA 5 crop, gộp logit", 5, I.aggregate_views(zc, "logit"), yv)
    pd.DataFrame(inf_rows)

In [ ]:
with K.guard("Bước 3: I04 độ phân giải kiểm tra"):
    res_rows = []
    for mult in (224, 256, 288, 320):
        size = int(round(IMG * mult / 224))
        try:
            _, _, z = I.predict_logits(model, vloader(size), DEV)
            res_rows.append({"exp_id": f"I04_{size}", "phương pháp": f"độ phân giải kiểm tra {size}", "mô hình": f"{FINAL_SRC} ({BB})",
                             "K": 1, **score(I.apply_temperature(z, 1.0), yv)})
        except Exception as e:
            res_rows.append({"exp_id": f"I04_{size}", "phương pháp": f"độ phân giải kiểm tra {size}", "K": 1,
                             "ghi chú": f"không áp dụng cho {BB}: {type(e).__name__}"})
    inf_rows += res_rows
    pd.DataFrame(res_rows)

In [ ]:
with K.guard("Bước 3: I05 ensemble, I06 EMA, I07 temperature scaling"):
    ens = K.top_backbones(backbones_tbl, 3)
    probs_list = []
    for e, n in ens:
        m = T.load_best(cfg(exp_id=e, backbone=n, seed=0), DEV)
        _, _, z = I.predict_logits(m, vloader(IMG), DEV); probs_list.append(I.apply_temperature(z, 1.0)); del m
    add("I05", "Ensemble " + "+".join(n for _, n in ens), len(ens), I.ensemble_probs(probs_list), yv,
        note="3 backbone tốt nhất ở Bước 1 (chọn theo val), trung bình xác suất")
    runs = TB.collect_summaries(OUT)
    t16 = runs[(runs.exp_id == "T16") & (runs.seed == 0)]
    if len(t16):
        r = t16.iloc[0]
        inf_rows.append({"exp_id": "I06", "phương pháp": "Trọng số EMA 0.99 (train có EMA)", "mô hình": f"T16 ({BB})", "K": 1,
                         "macro_f1": r.val_macro_f1, "top1": r.val_top1, "ece": r.val_ece,
                         "ghi chú": f"không tốn thêm khi suy luận; T00 cùng seed: macro-F1 {t00_ref:.4f}"})
    # I07: T khớp trên NỬA val (A), đo ECE trên nửa còn lại (B)
    rng = np.random.default_rng(0); idx = rng.permutation(len(yv)); a, b = idx[: len(idx) // 2], idx[len(idx) // 2:]
    Tcal = I.fit_temperature(z0[a], yv[a])
    pb_u, pb_c = I.apply_temperature(z0[b], 1.0), I.apply_temperature(z0[b], Tcal)
    print(f"T = {Tcal:.3f} | ECE nửa B trước {score(pb_u, yv[b])['ece']:.4f} -> sau {score(pb_c, yv[b])['ece']:.4f}")
    Tfull = I.fit_temperature(z0, yv)
    add("I07", "Temperature scaling (T khớp trên val)", 1, I.apply_temperature(z0, Tfull), yv,
        note=f"T={Tfull:.3f}; ECE nửa B: {score(pb_u, yv[b])['ece']:.4f} -> {score(pb_c, yv[b])['ece']:.4f} (T khớp trên nửa A)")
    pd.DataFrame(inf_rows)

In [ ]:
with K.guard("Bước 3: I08 gộp BN / FP16 và độ trễ"):
    fused = I.fuse_conv_bn(model)
    print("BN đã gộp:", fused.fuse_pairs, "cặp, sai số", fused.fuse_max_abs_err)
    _, _, zf = I.predict_logits(fused, vloader(IMG), DEV)
    add("I08a", "Gộp BN vào conv (FP32)", 1, I.apply_temperature(zf, 1.0), yv,
        note=f"{fused.fuse_pairs} cặp; max|Δ|={fused.fuse_max_abs_err:.1e}" + ("" if fused.fuse_pairs else " (backbone không có BatchNorm2d: không áp dụng)"))
    _, _, zh = I.predict_logits(model, vloader(IMG), DEV, amp=True)
    add("I08b", "AMP (autocast FP16)", 1, I.apply_temperature(zh, 1.0), yv)
    lat_rows = []
    for tag, m, dt, fusedflag in [("I00 FP32", model, "fp32", False), ("I08a gộp BN FP32", fused, "fp32", True),
                                  ("I08b AMP", model, "amp", False), ("FP16 .half()", model, "fp16", False)]:
        for batch in (1, 32):
            lat_rows.append({"cấu hình": tag, "bn_fused": fusedflag, **B.latency_report(m, batch, IMG, dt, DEV.type, 20, 100)})
    for k, nm in [(2, "I01 TTA lật"), (5, "I02a TTA 5 crop"), (10, "I02b TTA 5 crop+lật")]:
        lat_rows.append({"cấu hình": nm, "bn_fused": False, "batch": 1, **B.tta_latency(model, k, 1, IMG, "fp32", DEV.type)})
    lat_inf = pd.DataFrame(lat_rows); lat_inf.to_csv(f"{TAB}/latency_inference.csv", index=False)
    lat_inf

In [ ]:
with K.guard("Bước 3: tổng hợp, chọn cách suy luận chung kết"):
    LATMAP = {"I00": "I00 FP32", "I01": "I01 TTA lật", "I02a": "I02a TTA 5 crop", "I02b": "I02b TTA 5 crop+lật",
              "I03a": "I02a TTA 5 crop", "I03b": "I02a TTA 5 crop", "I08a": "I08a gộp BN FP32", "I08b": "I08b AMP"}
    def lat_of(eid, col):
        r = lat_inf[(lat_inf["cấu hình"] == LATMAP.get(eid)) & (lat_inf.batch == 1)]
        return float(r.iloc[0][col]) if len(r) else np.nan
    inference_tbl = pd.DataFrame(inf_rows)
    for col in ("p50", "p95", "p99"): inference_tbl[f"{col}_ms_b1"] = inference_tbl.exp_id.map(lambda e: lat_of(e, col))
    thr_row = lat_inf[(lat_inf["cấu hình"] == "I00 FP32") & (lat_inf.batch == 32)]
    inference_tbl["images_per_s"] = inference_tbl.exp_id.map(
        lambda e: (float(lat_inf[(lat_inf["cấu hình"] == LATMAP[e]) & (lat_inf.batch == 32)].images_per_s.iloc[0])
                   if e in LATMAP and len(lat_inf[(lat_inf["cấu hình"] == LATMAP[e]) & (lat_inf.batch == 32)]) else np.nan))
    p50_i00 = lat_of("I00", "p50")
    inference_tbl["chi phí tương đối vs I00"] = inference_tbl.exp_id.map(lambda e: lat_of(e, "p50") / p50_i00)
    inference_tbl.loc[inference_tbl["chi phí tương đối vs I00"].isna(), "chi phí tương đối vs I00"] = inference_tbl["K"]
    inference_tbl.to_csv(f"{TAB}/inference.csv", index=False)
    p95_by_id = {e: lat_of(e, "p95") for e in LATMAP if np.isfinite(lat_of(e, "p95"))}
    TTA, SPACE, TTA_SRC = K.choose_tta(inference_tbl, p95_by_id)
    K.log(f"suy luận chung kết (val, p95 batch-1 <= 100 ms): {TTA_SRC} tta={TTA} space={SPACE} | p95={p95_by_id.get(TTA_SRC)}")
    d = inference_tbl.dropna(subset=["macro_f1"]).copy(); d["lat_x"] = d["p50_ms_b1"].fillna(d["K"].fillna(1) * p50_i00)
    fig, ax = plt.subplots(figsize=(6, 4)); ax.scatter(d.lat_x, d.macro_f1)
    for _, r in d.iterrows(): ax.annotate(r.exp_id, (r.lat_x, r.macro_f1), fontsize=7)
    ax.set(xlabel="độ trễ batch-1 p50 (ms)", ylabel="macro-F1 val", title="Suy luận: độ chính xác vs độ trễ"); fig.tight_layout()
    fig.savefig(f"{TAB}/inference_tradeoff.png", dpi=130); plt.show()
    inference_tbl

## Bước 4 — Chung kết: ≥ 3 seed, test một lượt mỗi seed
Mốc `T00` + `I00`: chạy test từ đúng checkpoint đã chọn trên val (không huấn luyện lại). Cấu hình cuối `F01`: huấn luyện không chạm test,
rồi `final.final_predict` chạy test **một lượt** với mọi view TTA, ghi cả bản chưa hiệu chuẩn (`F01uncal`). Chạy lại sẽ bị chặn (`FileExistsError`).

In [ ]:
FINAL = dict(backbone=BB, **FINAL_RECIPE)
with K.guard("Bước 4: chung kết"):
    for s in SEEDS:
        K.stage(f"T00 test seed{s}", K.test_once_from_ckpt, cfg(exp_id="T00", backbone=BB, seed=s), DEV)   # mốc
        c = cfg(exp_id="F01", seed=s, **FINAL)
        K.stage(f"F01 train seed{s}", T.run, c)
        ok, r = K.stage(f"F01 final_predict seed{s}", FN.final_predict, c, tta=TTA, space=SPACE, calibrate=True, device=DEV)
        if ok: print(r)

In [ ]:
with K.guard("Bước 4: eval.py score/grade"):
    TEST_CSV, LAB = f"{LABELS_DIR}/test_subset0.csv", f"{LABELS_DIR}/labels.csv"
    os.makedirs(f"{SUB}/eval_out", exist_ok=True)
    for tag in ("F01", "F01uncal", "T00"):
        r = subprocess.run([sys.executable, f"{ROOT}/eval.py", "score", "--pred", f"{PRED}/{tag}_seed*_test.csv",
                            "--test-csv", TEST_CSV, "--labels", LAB, "--tag", tag, "--out", f"{SUB}/eval_out"],
                           capture_output=True, text=True)
        print(r.stdout, r.stderr[-500:])
    p95 = p95_by_id.get(TTA_SRC, p95_by_id.get("I00"))
    r = subprocess.run([sys.executable, f"{ROOT}/eval.py", "grade", "--final", f"{PRED}/F01_seed*_test.csv",
                        "--baseline", f"{PRED}/T00_seed*_test.csv", "--uncal", f"{PRED}/F01uncal_seed*_test.csv",
                        "--final-val", f"{PRED}/F01_seed*_val.csv", "--test-csv", TEST_CSV, "--labels", LAB,
                        "--latency-p95-ms", str(p95), "--out", f"{SUB}/eval_out"], capture_output=True, text=True)
    print(r.stdout, r.stderr[-500:])
    open(f"{SUB}/eval_out/grade.txt", "w").write(r.stdout + r.stderr)

### Phân tích lỗi (sau Bước 4)
Chỉ để phân tích và viết báo cáo: **không** quay lại sửa cấu hình rồi chạy test lần nữa.

In [ ]:
with K.guard("Phân tích lỗi"):
    names = ev.load_names(LAB); hard = ev.hard_class_indices(names)
    for tag in ("F01", "T00"):
        fig, cm, top = RF.confusion_figure(f"{PRED}/{tag}_seed*_test.csv", TEST_CSV, LAB, tag, f"{TAB}/confusion_{tag}.png"); plt.show()
        print(tag, "- các nhầm lẫn lớn nhất:"); print(top)
        top.to_csv(f"{TAB}/top_confusions_{tag}.csv", index=False)
    pair = (hard["Chinee Apple"], hard["Snake Weed"])
    RF.misclassified_grid(f"{PRED}/F01_seed0_test.csv", IMAGES_DIR, names, 12, pair, f"{TAB}/errors_chinee_snake.png"); plt.show()
    RF.misclassified_grid(f"{PRED}/F01_seed0_test.csv", IMAGES_DIR, names, 12, None, f"{TAB}/errors_most_confident.png"); plt.show()

## Bước 5 — Sản phẩm: `results.xlsx` (7 sheet), `curves/`, `predictions/`, bản ghi cấu hình đã chọn

In [ ]:
def write_results(extra=None):
    """Ghi results.xlsx (7 sheet bắt buộc + sheet phụ) và final_choice.json. Gọi lại được sau khi có thêm kết quả."""
    runs = TB.collect_summaries(OUT)
    per_class_tbl = RF.per_class_sheet({"F01 (cuối)": f"{PRED}/F01_seed*_test.csv", "T00 (mốc)": f"{PRED}/T00_seed*_test.csv"}, TEST_CSV, LAB)
    final_tbl = K.final_table(PRED, {"T00": "mốc: T00 + I00 (1 view)",
                                     "F01uncal": f"{BB} + {FINAL_SRC} + {TTA}/{SPACE}, chưa hiệu chuẩn",
                                     "F01": f"{BB} + {FINAL_SRC} {FINAL_RECIPE} + {TTA}/{SPACE} + temperature scaling"}, runs)
    sheets = {
        "Backbones": backbones_tbl[["exp_id", "backbone", "pretrained_tag", "params_m", "gmacs", "img_size", "epochs", "seed",
                                    "val_macro_f1", "val_top1", "sec_per_epoch", "lat_b1_fp32_ms", "ghi chú"]],
        "Training": training_tbl,
        "Inference": inference_tbl,
        "Final": final_tbl,
        "PerClass": per_class_tbl,
        "Latency": pd.concat([lat.assign(**{"cấu hình": lat.exp_id}), lat_inf], ignore_index=True),
    }
    lat_by_bb = lat[(lat.batch == 1) & (lat.dtype == "fp32")].groupby("backbone")["p50"].first()
    top = runs.sort_values("val_macro_f1", ascending=False).head(10)[["exp_id", "backbone", "seed", "val_macro_f1", "val_top1", "params_m", "gmacs", "sec_per_epoch"]].copy()
    top["lat_b1_fp32_p50_ms"] = top.backbone.map(lat_by_bb)
    top["ghi chú"] = np.where(top.exp_id == "T00", "mốc (công thức nền)", np.where(top.exp_id.isin(["T18", "F01", FINAL_SRC]), "cấu hình chung kết/tổ hợp", ""))
    sheets["Summary"] = top
    for name, df in (extra or {}).items(): sheets[name] = df
    TB.write_results_xlsx(f"{SUB}/results.xlsx", sheets, {"Backbones": "val_macro_f1", "Summary": "val_macro_f1"})
    choice = {"BEST_BB": BEST_BB, "noise_threshold": noise, "T00_val_f1": t00.val_macro_f1.tolist(), "winners": CHOSEN, "COMBO": COMBO,
              "best_single": [BEST_SINGLE[0], BEST_SINGLE[1]] if BEST_SINGLE else None, "FINAL_SRC": FINAL_SRC, "FINAL_RECIPE": FINAL_RECIPE,
              "TTA": TTA, "SPACE": SPACE, "TTA_SRC": TTA_SRC, "p95_by_id": p95_by_id, "SEEDS": list(SEEDS), "IMG": IMG, "EPOCHS": EPOCHS, "SMOKE": SMOKE,
              "ensemble": ens, "hours": hours(), "torch": torch.__version__, "timm": timm.__version__,
              "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(DEV)}
    json.dump(choice, open(f"{TAB}/final_choice.json", "w"), indent=1, default=str)
    have = {p.name.split("_")[0] for p in pathlib.Path(CURVE).glob("*.png")}
    print("đã ghi", f"{SUB}/results.xlsx", "| sheet:", list(sheets), "| thiếu ảnh curves:", sorted(set(runs.exp_id) - have))
    K.backup(SUB, DRIVE_OUT)
    return choice

with K.guard("Bước 5: results.xlsx"):
    print(json.dumps(write_results(), indent=1, default=str))

## Lab #2 theo slide (trang 78): ba cách khởi tạo × có/không CutMix, mean ± std qua 3 seed
Chạy SAU chung kết và sau khi đã ghi results.xlsx (nên nếu hết giờ GPU thì sản phẩm chính vẫn đủ). Chỉ dùng val, không dùng để chọn cấu hình chung kết.
Độ trễ p50/p95 trước và sau khi gộp BatchNorm hoặc FP16 nằm ở sheet Latency (I08).

In [ ]:
with K.guard("Lab #2 slide: khởi tạo x CutMix"):
    l2_seeds = SEEDS[:1] if SMOKE else SEEDS
    run_many([(e, dict(base, seed=s, **kw)) for s in l2_seeds for e, _, kw in K.LAB2_RUNS], budget_h=BUDGET_L2_H)
    runs = TB.collect_summaries(OUT)
    rows = []
    for e, desc, kw in K.LAB2_RUNS:
        r = runs[(runs.exp_id == e) & (runs.backbone == BB)]
        if r.empty: continue
        f1, t1 = r.val_macro_f1.to_numpy(), r.val_top1.to_numpy()
        sd = lambda a: float(np.std(a, ddof=1)) if len(a) > 1 else float("nan")
        rows.append({"exp_id": e, "cách khởi tạo": "từ đầu" if kw.get("init") == "scratch" else ("đóng băng" if kw.get("init") == "frozen" else "tinh chỉnh"),
                     "CutMix": "có" if kw.get("mix") == "cutmix" else "không", "n_seed": len(r), "seeds": sorted(r.seed.tolist()),
                     "top1_val_mean": float(t1.mean()), "top1_val_std": sd(t1), "macro_f1_val_mean": float(f1.mean()), "macro_f1_val_std": sd(f1),
                     "top1_val (mean ± std)": f"{t1.mean():.4f} ± {sd(t1):.4f}", "ghi chú": desc})
    lab2_tbl = pd.DataFrame(rows); lab2_tbl.to_csv(f"{TAB}/lab2_slide_matrix.csv", index=False)
    print(lab2_tbl.drop(columns=["top1_val_mean", "top1_val_std", "macro_f1_val_mean", "macro_f1_val_std"]).to_string())
    print(json.dumps(write_results({"Lab2_Slide": lab2_tbl}), indent=1, default=str)[:300])

In [ ]:
with K.guard("Dọn dẹp"):
    # các bước đã xong: xoá checkpoint (*.pt) để thư mục kết quả nhỏ gọn; giữ history/summary/logit làm bằng chứng
    print(f"đã xoá checkpoint: {K.prune_checkpoints(OUT):.0f} MB | tổng thời gian {hours():.2f} giờ")
    shutil.rmtree(f"{SUB}/code/__pycache__", ignore_errors=True)
    K.backup(SUB, DRIVE_OUT)
    K.log(f"HOÀN TẤT sau {hours():.2f} giờ")